In [7]:
from kiteconnect import KiteConnect
import pandas as pd
import datetime as dt
import os
import numpy as np
from stocktrends import Renko

In [9]:
request_token=open("request_token.txt","r").read()
key_secrets=open("api_key.txt","r").read().split()
kite = KiteConnect(api_key=key_secrets[0])
data = kite.generate_session(request_token=request_token,api_secret=key_secrets[1])

TokenException: Token is invalid or has expired.

In [4]:
# Get dump of all NSE instruments
instruments_dump=kite.instruments("NSE")
instruments_df=pd.DataFrame(instruments_dump)


In [5]:
def instrumentLookup(instruments_df,symbol):
    """Looks up instrument token for a given script from instrument dump"""
    try:
        return instruments_df[instruments_df.tradingsymbol==symbol].instrument_token.values[0]
    except:
        return -1


def fetchOHLC(ticker,interval,duration):
    """extracts historical data and outputs in the form of dataframe"""
    instrument = instrumentLookup(instruments_df,ticker)
    data = pd.DataFrame(kite.historical_data(instrument,dt.date.today()-dt.timedelta(duration), dt.date.today(),interval))
    data.set_index("date",inplace=True)
    return data

In [6]:
def maru_bozu(ohlc_df):    
    """returns dataframe with maru bozu candle column"""
    df = ohlc_df.copy()
    avg_candle_size = abs(df["close"] - df["open"]).median()
    df["h-c"] = df["high"]-df["close"]
    df["l-o"] = df["low"]-df["open"]
    df["h-o"] = df["high"]-df["open"]
    df["l-c"] = df["low"]-df["close"]
    df["maru_bozu"] = np.where((df["close"] - df["open"] > 2*avg_candle_size) & \
                               (df[["h-c","l-o"]].max(axis=1) < 0.005*avg_candle_size),"maru_bozu_green",
                               np.where((df["open"] - df["close"] > 2*avg_candle_size) & \
                               (abs(df[["h-o","l-c"]]).max(axis=1) < 0.005*avg_candle_size),"maru_bozu_red",False))
    df.drop(["h-c","l-o","h-o","l-c"],axis=1,inplace=True)
    return df

ohlc = fetchOHLC("ASIANPAINT","5minute",30)
maru_bozu_df = maru_bozu(ohlc)

In [7]:
ohlc

,open,high,low,close,volume
date,,,,,
2025-03-21 09:15:00+05:30,2291.65,2291.65,2278.45,2283.50,14804
2025-03-21 09:20:00+05:30,2283.10,2283.85,2277.85,2278.05,9578
2025-03-21 09:25:00+05:30,2278.15,2284.00,2278.15,2282.05,9619
2025-03-21 09:30:00+05:30,2282.65,2284.00,2279.55,2282.00,13174
2025-03-21 09:35:00+05:30,2282.40,2285.00,2281.10,2284.10,7125
...,...,...,...,...,...
2025-04-17 15:05:00+05:30,2465.10,2471.10,2465.00,2470.70,70724
2025-04-17 15:10:00+05:30,2470.70,2470.90,2469.00,2469.40,14947
2025-04-17 15:15:00+05:30,2469.20,2469.40,2464.90,2465.00,15791


In [8]:
maru_bozu_df

,open,high,low,close,volume,maru_bozu
date,,,,,,
2025-03-21 09:15:00+05:30,2291.65,2291.65,2278.45,2283.50,14804,False
2025-03-21 09:20:00+05:30,2283.10,2283.85,2277.85,2278.05,9578,False
2025-03-21 09:25:00+05:30,2278.15,2284.00,2278.15,2282.05,9619,False
2025-03-21 09:30:00+05:30,2282.65,2284.00,2279.55,2282.00,13174,False
2025-03-21 09:35:00+05:30,2282.40,2285.00,2281.10,2284.10,7125,False
...,...,...,...,...,...,...
2025-04-17 15:05:00+05:30,2465.10,2471.10,2465.00,2470.70,70724,False
2025-04-17 15:10:00+05:30,2470.70,2470.90,2469.00,2469.40,14947,False
2025-04-17 15:15:00+05:30,2469.20,2469.40,2464.90,2465.00,15791,False
